# Module 15 — Autoencoders for Anomaly Detection

> **Track 2 · Revolut Fintech Specialization**  
> **Difficulty**: Advanced  
> **Runtime**: ~6 minutes  
> **Key Libraries**: PyTorch, Scikit-learn, NumPy, Pandas, Plotly  
> **Prerequisite**: Module 12 (Isolation Forest), Module 14 (Weighted BCE)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Transaction Dataset](#3-synthetic-transaction-dataset)
4. [Autoencoder Architecture](#4-autoencoder-architecture)
5. [Training on Normal Transactions Only](#5-training-on-normal-transactions-only)
6. [Reconstruction Error as Anomaly Score](#6-reconstruction-error-as-anomaly-score)
7. [Threshold Selection](#7-threshold-selection)
8. [Evaluation Against Labels](#8-evaluation-against-labels)
9. [Comparison with Isolation Forest](#9-comparison-with-isolation-forest)
10. [Visualization Dashboard](#10-visualization-dashboard)
11. [Key Business Takeaway](#11-key-business-takeaway)

---
## §1 · Business Context

### Why Autoencoders for Anomaly Detection?

Autoencoders are **unsupervised neural networks** that learn to compress and reconstruct data.
For fraud detection, we train them **only on normal transactions**:

1. **Normal transactions** → autoencoder learns to reconstruct them well (low error).
2. **Anomalous transactions** → autoencoder fails to reconstruct them (high error).
3. **Reconstruction error** becomes the anomaly score.

| Advantage | Why It Matters |
|-----------|----------------|
| **Unsupervised** | No labeled fraud data needed |
| **Deep learning** | Captures complex non-linear patterns |
| **Scalable** | Handles millions of transactions with 100+ features |
| **Adaptable** | Retrain on recent "normal" data to track evolving patterns |

**Use cases at Revolut**:
- Detect **novel fraud patterns** not caught by rules or supervised models.
- Monitor **account takeover** (unusual login/transaction combos).
- Identify **money laundering** (structured transactions).

In this notebook we will:
1. Generate a 200K-transaction dataset with 30 features.
2. Build a PyTorch autoencoder with bottleneck architecture.
3. Train **only on normal transactions** (semi-supervised learning).
4. Use reconstruction error to score anomalies.
5. Compare against Isolation Forest from Module 12.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── PyTorch ──────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve,
    confusion_matrix
)

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")
print(f"  PyTorch {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

---
## §3 · Synthetic Transaction Dataset

We simulate **200,000 transactions** with **30 features** and a **2% fraud rate**.

In [ ]:
# ── Configuration ────────────────────────────────────────────────
N_TXN = 200_000
N_FRAUD = 4_000  # 2% fraud rate
N_LEGIT = N_TXN - N_FRAUD
N_FEATURES = 30

print(f"Generating transaction dataset:")
print(f"  Total: {N_TXN:,} transactions")
print(f"  Fraud: {N_FRAUD:,} ({N_FRAUD/N_TXN*100:.1f}%)")
print(f"  Features: {N_FEATURES}")

np.random.seed(42)

# ── Feature names (30 features) ──────────────────────────────────
feature_names = [
    "amount", "amount_log", "hour_of_day", "day_of_week", "is_weekend",
    "txn_count_1h", "txn_count_24h", "txn_count_7d",
    "amount_sum_1h", "amount_sum_24h", "amount_sum_7d",
    "amount_mean_7d", "amount_std_7d", "amount_max_7d",
    "days_since_last_txn", "merchant_risk_score", "country_risk_score",
    "is_international", "distance_from_home_km", "velocity_1h",
    "velocity_24h", "unique_merchants_7d", "unique_countries_7d",
    "device_mobile", "device_card", "is_atm", "is_transfer",
    "user_tenure_days", "credit_score", "z_score_amount",
]

# ── Generate legitimate transactions ─────────────────────────────
X_legit = np.random.randn(N_LEGIT, N_FEATURES)
X_legit[:, 0] = np.abs(np.random.lognormal(3.0, 1.0, N_LEGIT))
X_legit[:, 1] = np.log1p(X_legit[:, 0])
X_legit[:, 2] = np.random.randint(0, 24, N_LEGIT)
X_legit[:, 3] = np.random.randint(0, 7, N_LEGIT)
X_legit[:, 4] = (X_legit[:, 3] >= 5).astype(float)
X_legit[:, 5] = np.random.poisson(1, N_LEGIT)
X_legit[:, 6] = np.random.poisson(5, N_LEGIT)
X_legit[:, 7] = np.random.poisson(20, N_LEGIT)
X_legit[:, 15] = np.random.uniform(0, 50, N_LEGIT)
X_legit[:, 16] = np.random.uniform(0, 30, N_LEGIT)
X_legit[:, 17] = np.random.binomial(1, 0.15, N_LEGIT)
X_legit[:, 18] = np.random.exponential(50, N_LEGIT)
X_legit[:, 27] = np.random.exponential(365, N_LEGIT)
X_legit[:, 28] = np.random.normal(650, 80, N_LEGIT)
X_legit[:, 29] = np.random.normal(0, 1, N_LEGIT)

# ── Generate fraudulent transactions (anomalous) ─────────────────
X_fraud = np.random.randn(N_FRAUD, N_FEATURES)
X_fraud[:, 0] = np.abs(np.random.lognormal(5.0, 1.5, N_FRAUD))
X_fraud[:, 1] = np.log1p(X_fraud[:, 0])
X_fraud[:, 2] = np.random.choice([0, 1, 2, 3, 22, 23], N_FRAUD)
X_fraud[:, 5] = np.random.poisson(5, N_FRAUD)
X_fraud[:, 6] = np.random.poisson(15, N_FRAUD)
X_fraud[:, 15] = np.random.uniform(60, 100, N_FRAUD)
X_fraud[:, 16] = np.random.uniform(70, 100, N_FRAUD)
X_fraud[:, 17] = np.random.binomial(1, 0.60, N_FRAUD)
X_fraud[:, 18] = np.random.exponential(500, N_FRAUD)
X_fraud[:, 25] = np.random.binomial(1, 0.40, N_FRAUD)
X_fraud[:, 26] = np.random.binomial(1, 0.30, N_FRAUD)
X_fraud[:, 29] = np.random.normal(3, 1.5, N_FRAUD)

# ── Combine ──────────────────────────────────────────────────────
X = np.vstack([X_legit, X_fraud])
y = np.concatenate([np.zeros(N_LEGIT), np.ones(N_FRAUD)])

idx = np.random.permutation(N_TXN)
X = X[idx]
y = y[idx]

df = pd.DataFrame(X, columns=feature_names)
df["is_fraud"] = y

# Split: train on normal data only (semi-supervised)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Filter training set to normal transactions only
X_train_normal = X_train[y_train == 0]
print(f"\n✓ Training set (normal only): {len(X_train_normal):,} transactions")
print(f"  Test set: {len(X_test):,} ({int(y_test.sum()):,} fraud)")

# Scale features
scaler = StandardScaler()
X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train_normal_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)

train_ds = TensorDataset(X_train_t)
train_dl = DataLoader(train_ds, batch_size=512, shuffle=True)

print(f"\n✓ Data prepared for autoencoder training")

---
## §4 · Autoencoder Architecture

### Bottleneck Design

The autoencoder compresses input into a lower-dimensional "bottleneck"
and then reconstructs it. Anomalies have high reconstruction error
because they don't fit the learned "normal" pattern.

```
Input (30 features)
  ↓ [Encoder]
Bottleneck (8 features)  ← compressed representation
  ↓ [Decoder]
Output (30 features)     ← reconstructed input
```

In [ ]:
class FraudAutoencoder(nn.Module):
    """
    Autoencoder for anomaly detection.
    
    Architecture:
        Encoder: 30 → 128 → 64 → 32 → 8 (bottleneck)
        Decoder: 8 → 32 → 64 → 128 → 30
    """
    
    def __init__(self, input_dim, bottleneck_dim=8):
        super().__init__()
        
        # Encoder: compress input to bottleneck
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, bottleneck_dim),
        )
        
        # Decoder: reconstruct input from bottleneck
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim),
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
    def encode(self, x):
        """Get bottleneck representation."""
        return self.encoder(x)

# Initialize model
input_dim = X_train_normal_scaled.shape[1]
bottleneck_dim = 8

model = FraudAutoencoder(input_dim, bottleneck_dim)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"Bottleneck dimension: {bottleneck_dim}")
print(f"Compression ratio: {input_dim / bottleneck_dim:.1f}×")

---
## §5 · Training on Normal Transactions Only

### Key Insight

We train **only on normal transactions**. The autoencoder learns to
reconstruct normal patterns well. Anomalies (fraud) will have high
reconstruction error because they don't fit the learned pattern.

In [ ]:
# Training configuration
n_epochs = 50
learning_rate = 0.001
criterion = nn.MSELoss()  # reconstruction error
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Training Autoencoder:")
print(f"  Epochs: {n_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Training samples: {len(X_train_normal):,} (normal only)")
print(f"  Loss function: MSE (reconstruction error)")

# Training loop
model.train()
train_losses = []

t0 = time.time()
for epoch in range(n_epochs):
    epoch_loss = 0
    n_batches = 0
    
    for (X_batch,) in train_dl:
        optimizer.zero_grad()
        
        # Forward pass: reconstruct input
        reconstructed = model(X_batch)
        
        # Compute reconstruction error
        loss = criterion(reconstructed, X_batch)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}/{n_epochs}: loss = {avg_loss:.6f}")

train_time = time.time() - t0
print(f"\n✓ Trained in {train_time:.2f}s")
print(f"  Final training loss: {train_losses[-1]:.6f}")

In [ ]:
# Plot training loss curve
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=train_losses,
    mode="lines",
    name="Training Loss",
    line=dict(color="#636EFA", width=2.5),
))

fig.update_layout(
    title="Autoencoder Training Loss (MSE Reconstruction Error)",
    xaxis_title="Epoch",
    yaxis_title="MSE Loss",
    height=400,
)
fig.show()

print("💡 Loss decreases as the autoencoder learns to reconstruct normal transactions")
print("   Convergence indicates the model has learned the 'normal' pattern")

---
## §6 · Reconstruction Error as Anomaly Score

### Compute Reconstruction Error on Test Set

In [ ]:
# Compute reconstruction error for all test samples
model.eval()
with torch.no_grad():
    # Reconstruct test samples
    X_test_reconstructed = model(X_test_t)
    
    # Compute per-sample MSE (reconstruction error)
    reconstruction_errors = torch.mean((X_test_t - X_test_reconstructed) ** 2, dim=1)
    reconstruction_errors = reconstruction_errors.numpy()

df_test = pd.DataFrame(X_test, columns=feature_names)
df_test["is_fraud"] = y_test
df_test["reconstruction_error"] = reconstruction_errors

print(f"Reconstruction Error Statistics:")
print(f"  Legit (n={int((y_test == 0).sum()):,}):")
print(f"    Mean : {df_test[df_test['is_fraud']==0]['reconstruction_error'].mean():.6f}")
print(f"    Median: {df_test[df_test['is_fraud']==0]['reconstruction_error'].median():.6f}")
print(f"    Std  : {df_test[df_test['is_fraud']==0]['reconstruction_error'].std():.6f}")

print(f"\n  Fraud (n={int((y_test == 1).sum()):,}):")
print(f"    Mean : {df_test[df_test['is_fraud']==1]['reconstruction_error'].mean():.6f}")
print(f"    Median: {df_test[df_test['is_fraud']==1]['reconstruction_error'].median():.6f}")
print(f"    Std  : {df_test[df_test['is_fraud']==1]['reconstruction_error'].std():.6f}")

print(f"\n💡 Fraudulent transactions have HIGHER reconstruction error")
print(f"   because they don't fit the learned 'normal' pattern")

In [ ]:
# Visualize reconstruction error distribution
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df_test[df_test["is_fraud"] == 0]["reconstruction_error"],
    name="Legitimate",
    nbinsx=100,
    marker_color="#00CC96",
    opacity=0.7,
))

fig.add_trace(go.Histogram(
    x=df_test[df_test["is_fraud"] == 1]["reconstruction_error"],
    name="Fraud",
    nbinsx=100,
    marker_color="#EF553B",
    opacity=0.7,
))

fig.update_layout(
    title="Reconstruction Error Distribution (Legit vs. Fraud)",
    xaxis_title="Reconstruction Error (MSE)",
    yaxis_title="Count",
    barmode="overlay",
    height=440,
)
fig.show()

print("💡 Legitimate transactions cluster at low reconstruction error")
print("   Fraudulent transactions have higher error (anomalous)")
print("   The separation allows threshold-based anomaly detection")

---
## §7 · Threshold Selection

### Percentile-Based Threshold

We select the threshold at the **98th percentile** of reconstruction error
(matching the 2% fraud rate in the test set).

In [ ]:
# Compute threshold at 98th percentile (matching 2% fraud rate)
fraud_rate_test = y_test.mean()
threshold_percentile = (1 - fraud_rate_test) * 100
threshold = np.percentile(reconstruction_errors, threshold_percentile)

print(f"Threshold Selection:")
print(f"  Fraud rate in test set: {fraud_rate_test*100:.2f}%")
print(f"  Percentile: {threshold_percentile:.2f}")
print(f"  Threshold: {threshold:.6f}")

# Predict anomalies
y_pred = (reconstruction_errors >= threshold).astype(int)
print(f"\nPredictions:")
print(f"  Anomalies flagged: {y_pred.sum():,}")
print(f"  Actual fraud: {int(y_test.sum()):,}")

# Metrics
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"\nPerformance:")
print(f"  Precision: {prec:.4f}")
print(f"  Recall: {rec:.4f}")
print(f"  F1-Score: {f1:.4f}")

print(f"\n💡 Threshold at {threshold_percentile:.1f}th percentile catches ~{rec*100:.0f}% of fraud")
print(f"   with {prec*100:.1f}% precision")

---
## §8 · Evaluation Against Labels

In [ ]:
# Full evaluation metrics
roc_auc = roc_auc_score(y_test, reconstruction_errors)
pr_auc = average_precision_score(y_test, reconstruction_errors)

print("=" * 70)
print("AUTOENCODER EVALUATION (Test Set)")
print("=" * 70)
print(f"\nPrecision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"  True Negatives : {cm[0, 0]:>8,}")
print(f"  False Positives: {cm[0, 1]:>8,}")
print(f"  False Negatives: {cm[1, 0]:>8,}")
print(f"  True Positives : {cm[1, 1]:>8,}")

---
## §9 · Comparison with Isolation Forest

### Train Isolation Forest on Same Data

In [ ]:
# Train Isolation Forest
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=fraud_rate_test,
    random_state=42,
    n_jobs=-1,
)

print("Training Isolation Forest …")
t0 = time.time()
iso_forest.fit(X_train_normal_scaled)
iso_time = time.time() - t0
print(f"✓ Trained in {iso_time:.2f}s")

# Predict
iso_predictions = iso_forest.predict(X_test_scaled)
iso_anomaly = (iso_predictions == -1).astype(int)
iso_scores = -iso_forest.decision_function(X_test_scaled)  # higher = more anomalous

# Metrics
iso_prec = precision_score(y_test, iso_anomaly)
iso_rec = recall_score(y_test, iso_anomaly)
iso_f1 = f1_score(y_test, iso_anomaly)
iso_roc_auc = roc_auc_score(y_test, iso_scores)
iso_pr_auc = average_precision_score(y_test, iso_scores)

print(f"\nIsolation Forest Metrics:")
print(f"  Precision: {iso_prec:.4f}")
print(f"  Recall   : {iso_rec:.4f}")
print(f"  F1-Score : {iso_f1:.4f}")
print(f"  ROC-AUC  : {iso_roc_auc:.4f}")
print(f"  PR-AUC   : {iso_pr_auc:.4f}")

In [ ]:
# Compile comparison
comparison = pd.DataFrame({
    "Model": ["Autoencoder", "Isolation Forest"],
    "Precision": [prec, iso_prec],
    "Recall": [rec, iso_rec],
    "F1-Score": [f1, iso_f1],
    "ROC-AUC": [roc_auc, iso_roc_auc],
    "PR-AUC": [pr_auc, iso_pr_auc],
    "Training Time (s)": [train_time, iso_time],
}).round(4)

print("=" * 90)
print("MODEL COMPARISON: Autoencoder vs. Isolation Forest")
print("=" * 90)
print(comparison.to_string(index=False))

print("\n💡 Key observations:")
print("   - Autoencoder often has higher PR-AUC (better for imbalanced data)")
print("   - Isolation Forest is faster to train")
print("   - Autoencoder captures non-linear patterns better")
print("   - Isolation Forest is more interpretable")

---
## §10 · Visualization Dashboard

In [ ]:
# ── 10.1 Precision-Recall Curves ─────────────────────────────────
fig = go.Figure()

# Autoencoder
prec_curve, rec_curve, _ = precision_recall_curve(y_test, reconstruction_errors)
fig.add_trace(go.Scatter(
    x=rec_curve, y=prec_curve,
    mode="lines",
    name=f"Autoencoder (PR-AUC = {pr_auc:.4f})",
    line=dict(color="#EF553B", width=2.5),
))

# Isolation Forest
prec_iso, rec_iso, _ = precision_recall_curve(y_test, iso_scores)
fig.add_trace(go.Scatter(
    x=rec_iso, y=prec_iso,
    mode="lines",
    name=f"Isolation Forest (PR-AUC = {iso_pr_auc:.4f})",
    line=dict(color="#636EFA", width=2.5),
))

fig.update_layout(
    title="Precision-Recall Curves (Higher is Better)",
    xaxis_title="Recall",
    yaxis_title="Precision",
    height=500,
    legend=dict(orientation="h", y=1.12),
)
fig.show()

print("💡 PR-AUC is the gold standard for imbalanced anomaly detection")
print("   Autoencoder often outperforms Isolation Forest on complex patterns")

In [ ]:
# ── 10.2 ROC Curves ──────────────────────────────────────────────
fig = go.Figure()

# Autoencoder
fpr_ae, tpr_ae, _ = roc_curve(y_test, reconstruction_errors)
fig.add_trace(go.Scatter(
    x=fpr_ae, y=tpr_ae,
    mode="lines",
    name=f"Autoencoder (AUC = {roc_auc:.4f})",
    line=dict(color="#EF553B", width=2.5),
))

# Isolation Forest
fpr_iso, tpr_iso, _ = roc_curve(y_test, iso_scores)
fig.add_trace(go.Scatter(
    x=fpr_iso, y=tpr_iso,
    mode="lines",
    name=f"Isolation Forest (AUC = {iso_roc_auc:.4f})",
    line=dict(color="#636EFA", width=2.5),
))

# Random baseline
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Random Classifier",
    line=dict(color="gray", width=1, dash="dash"),
))

fig.update_layout(
    title="ROC Curves",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=500,
    legend=dict(orientation="h", y=1.12),
)
fig.show()

In [ ]:
# ── 10.3 Business Impact ─────────────────────────────────────────
COST_FN = 2500  # £ per missed fraud
COST_FP = 5     # £ per false alarm

def compute_cost(y_true, y_pred, cost_fn=COST_FN, cost_fp=COST_FP):
    cm = confusion_matrix(y_true, y_pred)
    fn = cm[1, 0]
    fp = cm[0, 1]
    tp = cm[1, 1]
    return fn * cost_fn + fp * cost_fp, tp, fp, fn

cost_ae, tp_ae, fp_ae, fn_ae = compute_cost(y_test, y_pred)
cost_iso, tp_iso, fp_iso, fn_iso = compute_cost(y_test, iso_anomaly)

print("=" * 90)
print("BUSINESS IMPACT COMPARISON (Test Set, 60K transactions)")
print("=" * 90)
print(f"\nAssumptions:")
print(f"  Cost per missed fraud (FN): £{COST_FN:,}")
print(f"  Cost per false alarm (FP) : £{COST_FP}")

print(f"\n{'Model':<20} {'TP':>6} {'FP':>8} {'FN':>8} {'Total Cost (£)':>15}")
print("-" * 70)
print(f"{'Autoencoder':<20} {tp_ae:>6} {fp_ae:>8,} {fn_ae:>8} £{cost_ae:>14,}")
print(f"{'Isolation Forest':<20} {tp_iso:>6} {fp_iso:>8,} {fn_iso:>8} £{cost_iso:>14,}")

print(f"\n💡 Autoencoder has lower total cost due to better fraud detection")
print(f"   Savings: £{cost_iso - cost_ae:,.0f}")

---
## §11 · Key Business Takeaway

> ### 🎯 Action Items for a Fintech Data Scientist
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Novel fraud patterns, no labeled data, complex non-linear patterns |
> | **Training** | Train only on normal transactions (semi-supervised) |
> | **Architecture** | Bottleneck design (30 → 8 → 30) for compression |
> | **Threshold** | Percentile-based (matching expected fraud rate) |
> | **Evaluation** | PR-AUC (not accuracy) for imbalanced anomaly detection |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Autoencoders complement Isolation Forest**:
>    - Isolation Forest: fast, interpretable, good baseline.
>    - Autoencoder: captures non-linear patterns, higher PR-AUC.
>
> 2. **Retrain weekly on recent "normal" data**:
>    - Fraud patterns evolve — the model must adapt.
>    - Use the last 30 days of "normal" transactions (no fraud flags).
>
> 3. **Bottleneck dimension is critical**:
>    - Too small: loses information (underfitting).
>    - Too large: memorizes noise (overfitting).
>    - Start with `input_dim / 4` (e.g., 8 for 30 features).
>
> 4. **Reconstruction error threshold**:
>    ```python
>    # Set threshold at (1 - fraud_rate) percentile
>    fraud_rate = 0.02  # expected 2% fraud
>    threshold = np.percentile(train_errors, (1 - fraud_rate) * 100)
>    ```
>
> 5. **Monitor reconstruction error distribution**:
>    - If the distribution shifts, fraud patterns may be changing.
>    - Retrain the model on recent data.
>
> ### 📌 Autoencoder Cheat Sheet
>
> ```python
> class Autoencoder(nn.Module):
>     def __init__(self, input_dim, bottleneck_dim):
>         super().__init__()
>         self.encoder = nn.Sequential(
>             nn.Linear(input_dim, 128), nn.ReLU(),
>             nn.Linear(128, 64), nn.ReLU(),
>             nn.Linear(64, bottleneck_dim),
>         )
>         self.decoder = nn.Sequential(
>             nn.Linear(bottleneck_dim, 64), nn.ReLU(),
>             nn.Linear(64, 128), nn.ReLU(),
>             nn.Linear(128, input_dim),
>         )
>     
>     def forward(self, x):
>         return self.decoder(self.encoder(x))
>
> # Train on normal data only
> model.train()
> for X_batch in train_dl:
>     loss = nn.MSELoss()(model(X_batch), X_batch)
>     loss.backward()
>     optimizer.step()
> ```
>
> ### 🔑 Key Takeaways
>
> 1. **Autoencoders are unsupervised** — no labeled fraud data needed.
> 2. **Train on normal data only** — anomalies have high reconstruction error.
> 3. **Bottleneck architecture** forces the model to learn compressed representations.
> 4. **PR-AUC > ROC-AUC** for rare anomalies.
> 5. **Autoencoders capture non-linear patterns** that Isolation Forest may miss.